# A LoKI batch

This notebook tells how a batch of reductions is run in the architecture sketch,
on the same LoKI@Larmor tutorial files as `loki-session.ipynb`. That notebook is
one person moving one parameter; this one is many samples reduced with the same
settings, first by a person and then as the runs arrive.

Four samples were measured, each with a transmission run just before it, against
one solvent background and one empty-beam run. Reducing all four to I(Q) is the
job an ISIS batch file does.

Each half starts with the obvious program: the workflow and a `for` loop over
`client.run`. Records, provenance, and labels come from that alone. The loop is
then broken once per half, and the batch vocabulary is introduced as the answer
to each break:

- A **template** is a stored, versioned partial request: every parameter except
  the ones that change per sample, which are its **blanks**.
- The **pinned** values of a request are what a person set for one member
  beyond the template; a template change does not move them.
- A **batch** is the records under one label, one **member** per sample. Nothing
  else is stored: the batch table is a query over the records.
- A **rule** is a template plus a **lookup** that fills blanks per dataset, a
  **selector** that says which datasets it applies to, and a **bound** below which
  it does not fire on its own.
- `apply` is the one operation that makes requests from a template or rule. The
  trigger loop, `backlog`, and `reprocess` all call it.

Tables are pandas frames throughout; the design document
`docs/developer/rules.md` has a section that maps each concept to its pandas
counterpart.

## The journal and the arriving runs

The tutorial files carry run numbers but no sample name or run role, which a rule
needs to tell a sample from a transmission run. At a facility the catalogue
declares these fields. Here a journal frame stands in for it and is handed to the
folder source, which attaches the fields to each dataset by run number.

The folder the source reads is a temporary one, and `arrive` links tutorial files
into it, so that the notebook can let runs arrive while a rule is active. The
ISIS polymer and its transmission run are held back for that.

In [ ]:
import tempfile
from pathlib import Path

import pandas as pd
import plopp as pp

from ess.apps import loki
from ess.apps.batch import (
    TriggerLoop,
    apply,
    backlog,
    batch_table,
    dataset_table,
    reprocess,
    trigger_status,
)
from ess.apps.client import local
from ess.apps.rules import (
    AsOf,
    Bound,
    Like,
    Lookup,
    LookupEntry,
    Rule,
    Selector,
    Template,
)
from ess.apps.sources import FolderSource
from ess.apps.spec import dataset_ref
from ess.reduce.spec.parameters import QEdges

journal = pd.DataFrame.from_records(
    [
        (60384, 'porous silica', 'transmission'),
        (60385, 'porous silica', 'sample'),
        (60386, 'AgBeh', 'transmission'),
        (60387, 'AgBeh', 'sample'),
        (60388, 'deuterated SDS', 'transmission'),
        (60389, 'deuterated SDS', 'sample'),
        (60392, 'empty beam', 'empty-beam'),
        (60393, 'solvent', 'background'),
        (60394, 'ISIS polymer', 'transmission'),
        (60395, 'ISIS polymer', 'sample'),
    ],
    columns=['run', 'sample', 'role'],
).set_index('run')
journal

In [ ]:
cache = loki.cache()
root = Path(tempfile.mkdtemp(prefix='loki-batch-'))
incoming = root / 'incoming'
incoming.mkdir()


def arrive(*runs):
    """Let runs arrive: link their tutorial files into the folder the source reads."""
    for run in runs:
        (path,) = cache.glob(f'{run}-*.nxs')
        (incoming / path.name).symlink_to(path)


arrive(*journal.index.drop([60394, 60395]))

client = local(
    root / 'store',
    instrument='loki',
    proposal='p1',
    submitter='notebook',
    registry=loki.registry(),
    sources=[
        FolderSource(
            incoming,
            '*.nxs',
            identity=r'(?P<run>\d+)-.*',
            instrument='loki',
            journal=journal.to_dict('index'),
        )
    ],
)

dataset_table(client)

## A batch by hand

The beam centre is computed once, from the AgBeh run, as in `loki-session.ipynb`.
Then the obvious program: the values the samples share in one dict, and a loop
that runs the workflow once per sample. The label and the member key put the
three records in one batch, and plotting the batch is plotting a dict of record
outputs.

In [ ]:
def run(number):
    return dataset_ref(instrument='loki', run=number)


center = client.run(loki.BEAM_CENTER, {'sample_run': run(60387)})

shared = {
    'background_run': run(60393),
    'background_transmission_run': run(60392),
    'empty_beam_run': run(60392),
    'direct_beam': dataset_ref(path=cache / 'direct-beam-loki-all-pixels.h5'),
    'beam_center': center.ref(),
    'q': QEdges(start=0.01, stop=0.3, num_bins=100),
}
samples = {
    'porous silica': (60385, 60384),
    'AgBeh': (60387, 60386),
    'deuterated SDS': (60389, 60388),
}
for name, (sample, transmission) in samples.items():
    client.run(
        loki.IOFQ,
        {
            **shared,
            'sample_run': run(sample),
            'sample_transmission_run': run(transmission),
        },
        label='samples',
        member_key=name,
    )

In [ ]:
def curves(label, names=None):
    """The I(Q) of every member of a batch, keyed by member or by a name for it."""
    names = names or {}
    return {
        names.get(r.request.member_key, r.request.member_key): client.output(r, 'iofq')
        for r in client.batch(label)
    }


pp.plot(curves('samples'), norm='log')

### Correcting one member

The deuterated SDS curve wants a lower Q edge. The loop's answer is one more
`client.run` under the same label and member key. The new record supersedes the
earlier one in the batch, and the earlier one is still in the store. So far
nothing is missing.

In [ ]:
client.run(
    loki.IOFQ,
    {
        **shared,
        'sample_run': run(60389),
        'sample_transmission_run': run(60388),
        'q': QEdges(start=0.005, stop=0.3, num_bins=100),
    },
    label='samples',
    member_key='deuterated SDS',
)


def q_table(label):
    """The resolved Q binning of every member of a batch."""
    return pd.DataFrame(
        {r.request.member_key: r.resolved_params['q'] for r in client.batch(label)}
    ).T


q_table('samples')

### Where the loop breaks

The instrument scientist then changes the default Q binning to 200 bins. Which
member keeps its Q range? The records hold the resolved values only. The table
above cannot say whether 0.005 was chosen or was the default of its day, nor
whether the 100 bins next to it were chosen with it.

The loop's fix is a second dict of per-member overrides beside `shared`, and a
branch that applies it. That is the right fix, and it names what the loop lacks:
`shared` is a template, and the overrides are the pinned values. What the loop
still cannot do is make the record cite them, so that the split outlives the
notebook.

### The template

The template holds what `shared` held, under a name and a version. The two
blanks are the sample run and its transmission run. `dataset_field` names the
blank a dataset fills when a rule applies the template to one; it matters only
in the second half.

In [ ]:
template = Template(
    name='loki-iofq-larmor',
    spec=loki.IOFQ.id,
    params=shared,
    blanks=('sample_run', 'sample_transmission_run'),
    dataset_field='sample_run',
)
template.id, template.blanks

The person fills the blanks in a table, one row per member, keyed by a name they
choose, plus a cell for the one Q range that is not the default. This is the
ISIS batch file, and a blank cell falls through to the template. `apply` fills
the template once per row and returns the requests as a group; nothing exists in
the record store yet.

Validation runs per request before anything is submitted: the schema, the
parameter values, and whether every referenced dataset and record can be found.
A form would show these errors next to the row.

In [ ]:
table = pd.DataFrame(
    {
        'sample_run': [run(60385), run(60387), run(60389)],
        'sample_transmission_run': [run(60384), run(60386), run(60388)],
        'q': [None, None, QEdges(start=0.005, stop=0.3, num_bins=100)],
    },
    index=['porous silica', 'AgBeh', 'deuterated SDS'],
)
group = apply(client, template, pinned=table, label='samples')
pd.DataFrame(
    {key: client.validate(request).model_dump() for key, request in group.items()}
).T

The group is submitted under the same label, so its records supersede the
loop's. The batch table is what was reduced with which values: one row per
member, its latest record, the template version that filled it, and the values
of the fields that differ per member. Those are the template's blanks and every
field a member pinned. The `pinned` column answers the question above: it names
the fields of the row a person chose. A field that holds a model, such as `q`,
is one column per leaf. The table is computed from the records each time it is
asked for.

In [ ]:
records = client.submit_group(group)
batch_table(client, 'samples')

## Automatic reduction

Typing the table by hand does not scale to a beamtime. The obvious program
again: a loop over the datasets the source knows, which reduces every sample run
after a chosen run number, pairs it with the nearest transmission run before it,
and remembers what it has done so that the next pass does not reduce it again.
`fire` is what a scheduler would call every minute. The bound is the newest run
when the loop was written, so that it does not reduce the whole beamtime by
surprise.

In [ ]:
BOUND = 60393


def candidates():
    """The sample runs after the bound."""
    return [
        d for d in client.datasets() if d.fields['role'] == 'sample' and d.run > BOUND
    ]


def nearest(role, before):
    """The nearest dataset before `before` whose journal role is `role`."""
    return max(
        (d for d in client.datasets() if d.fields['role'] == role and d.run < before.run),
        key=lambda d: d.run,
    )


def reduce(d):
    """Reduce one sample run, paired with the nearest transmission run before it."""
    return client.run(
        loki.IOFQ,
        {
            **shared,
            'sample_run': d.ref,
            'sample_transmission_run': nearest('transmission', d).ref,
        },
        label='auto',
        member_key=str(d.ref),
    )


seen = set()


def fire():
    for d in candidates():
        if d.ref not in seen:
            reduce(d)
            seen.add(d.ref)


arrive(60394, 60395)
fire()
[str(r.request.member_key) for r in client.batch('auto')]

### Where the loop breaks

The backend restarts overnight, and `seen` is gone. Clearing it here stands in
for the restart. The next pass reduces the polymer again.

In [ ]:
seen.clear()
fire()
[(r.id, r.created) for r in client.records(label='auto')]

Writing `seen` to disk does not help. It is then a second truth beside the
records, and the two disagree whenever the process dies between the two writes.
The fix is to ask the records instead. The loop then keeps nothing, and a pass
after a restart fires on nothing.

In [ ]:
def fire():
    for d in candidates():
        if client.latest('auto', member_key=str(d.ref)) is None:
            reduce(d)


fire()
len(client.records(label='auto'))

### The rule

What is left is a loop that keeps nothing, and every line of it is a decision
that nobody at the instrument writes in Python: which datasets (`candidates`),
from when (`BOUND`), which transmission run (`nearest`), and which values
(`shared`). A UI cannot show them, an instrument scientist cannot edit them, and
a record cannot cite which version of them made it. A rule holds the four as
data:

- The **selector** picks the datasets whose journal role is `sample`, after the
  **bound**.
- The **lookup** fills the transmission run. Its one entry is an *as-of* fill:
  the nearest dataset before the member whose role is `transmission`. It is
  resolved against the member's own run number, not against the time of
  submission, so a late or repeated reduction gets the same transmission run.
- The **template** holds the shared values, and its `dataset_field` says that
  the selected dataset fills `sample_run`.

The rule's records go under its own name; the loop's batch stays as it is.

In [ ]:
rule = Rule(
    name='iofq-auto',
    template=template,
    lookup=Lookup(
        name='transmission',
        entries=(
            LookupEntry(
                name='nearest-transmission',
                fills={
                    'sample_transmission_run': AsOf(
                        match={'role': Like(pattern='transmission')}
                    )
                },
            ),
        ),
    ),
    selector=Selector(match={'role': Like(pattern='sample')}, after=Bound(run=BOUND)),
)
rule.id

`trigger_status` is the trigger loop's decision for one dataset, and why. It is
the same function the loop calls, so what a person reads is what the loop does.
The polymer is the one dataset the loop fires on: a sample run after the bound
with no record under the rule's label. The three samples at or before the bound
are the backlog.

In [ ]:
def trigger_table(rule):
    """The trigger status of every dataset the source knows."""
    return dataset_table(client).join(
        pd.DataFrame(
            [
                {'dataset': str(d.ref), **trigger_status(client, rule, d).model_dump()}
                for d in client.datasets()
            ]
        ).set_index('dataset')
    )


loop = TriggerLoop(client, rule)
trigger_table(rule)

In [ ]:
fired = loop.run_once()
[(r.request.member_key, r.status.value) for r in fired]

The loop keeps no memory, for the same reason the fixed loop above keeps none.
Every clause of the trigger status is a query over the records and the source,
so a second pass, or a pass after a restart, fires on nothing.

In [ ]:
loop.run_once()

### The backlog

The samples before the bound are offered as a backlog when the rule is created.
`backlog` returns a group like `apply` does, and nothing runs until the person
submits it. The rule's member keys are dataset identities, not names a person
chose. `dataset_table` is keyed by the same identities, so joining it onto the
batch table shows which sample each row is.

A rule pins nothing, so `pinned` is empty in every row. The blanks show what the
rule filled: `sample_run` is the selected dataset, and `sample_transmission_run`
is what the as-of fill found, each sample's own transmission run.

In [ ]:
def with_samples(table):
    """A rule's batch table with the sample name of each member."""
    samples = dataset_table(client)['sample']
    return table.join(samples).set_index('sample', append=True)


client.submit_group(backlog(client, rule))
with_samples(batch_table(client, rule))

### A correction, then a new rule version

A person corrects one member of the rule's batch as in the batch by hand: `apply`
on the rule for that one dataset, with the corrected value pinned. The new record
supersedes the rule's under the same member key.

In [ ]:
polymer = next(d for d in client.datasets() if d.run == 60395)
client.submit_group(
    apply(
        client,
        rule,
        [polymer],
        {str(polymer.ref): {'q': QEdges(start=0.005, stop=0.3, num_bins=100)}},
    )
)
with_samples(batch_table(client, rule))

Now the change the first half asked about: the instrument scientist moves the
default Q binning to 200 bins. That is a new template version and so a new rule
version; the old versions stay, and each record says which version made it.

`reprocess` offers the members whose latest record came from an older rule
version. It keeps what a person pinned and fills everything else from the new
versions, which is what the loop with an overrides dict could not do for records
it had already made. A pinned value replaces its field whole: the `q` pinned for
the polymer moved the lower edge, and it also pins the 100 bins. The table shows
that, and whether the pinned value still stands is the person's call.

In [ ]:
rule_v2 = rule.revise(
    template=template.revise(q=QEdges(start=0.01, stop=0.3, num_bins=200))
)
client.submit_group(reprocess(client, rule_v2))
with_samples(batch_table(client, rule_v2))

The history of one member is an ordinary record query. The polymer has three
records under the rule's label: the loop's, the correction, and the reprocess.
The latest one is the batch table's row.

In [ ]:
pd.DataFrame(
    [
        {
            'record': r.id,
            'rule': r.request.submission.rule,
            'pinned': list(r.request.submission.pinned),
            'q_start': r.resolved_params['q']['start'],
            'q_bins': r.resolved_params['q']['num_bins'],
        }
        for r in client.records(label='iofq-auto', member_key=str(polymer.ref))
    ]
)

In [ ]:
names = {str(run(n)): s for n, s in journal['sample'].items()}
pp.plot(curves('iofq-auto', names), norm='log')